In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
import random
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import lightgbm as lgb
import pickle

# 変数の設定

In [ ]:
class CFG:
    VER = 1
    AUTHOR = "suba"
    COMPETITION = "atmacup17"
    DATA_PATH = Path("/workspace/kaggle_llm_book/ch3/data/takaito/atmacup17")
    MODEL_NAME = "lightgbm"
    SEED = 42
    N_SPLIT = 3
    TARGET_COL = "Recommended IND"
    TARGET_COL_CLASS_NUM = 2
    METRIC = "auc"
    METRIC_MAXIMIZE_FLAG = True
    NUM_BOOST_ROUND = 10000
    EARLY_STOPPING_ROUND = 100
    VERBOSE = 250
    LGB_PARAMS= {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": 0.01,
        "seed": SEED,
    }
    NGRAM_RANGE = (1, 2)
    LOWERCASE = True
    SUBLINEAR_TF = True
    MAX_DF = 0.8
    MIN_DF = 5
    SVD_DIM = 256
    PREFIX  = f"{AUTHOR}_{MODEL_NAME}_seed{SEED}_ver{VER}"

# 乱数の設定

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
seed_everything(CFG.SEED)

# データの読み込みと前処理

In [ ]:
clothing_master_df = pd.read_csv(CFG.DATA_PATH / "clothing_master.csv")
train_df = pd.read_csv(CFG.DATA_PATH / "train.csv")
test_df = pd.read_csv(CFG.DATA_PATH / "test.csv")

In [ ]:
def make_text_column(df):
    df["text"] = df["Title"] + " " + df["Review Text"]
    return df

def preprocessing(df, clothing_master_df):
    df["Title"] = df["Title"].fillna("")
    df["Review Text"] = df["Review Text"].fillna("")
    df = df.merge(clothing_master_df, on="Clothing ID", how="left")
    df = make_text_column(df)
    return df

In [ ]:
train_df = preprocessing(train_df, clothing_master_df)
test_df = preprocessing(test_df, clothing_master_df)

In [ ]:
train_df.head()